# LeetCode 820 — Short Encoding of Words bằng Python

Link đề: https://leetcode.com/problems/short-encoding-of-words/ · Độ khó: **Medium** · Chủ đề: String, Hash Table, Trie (cây tiền tố), Sorting

Skeleton LeetCode:

```python
class Solution:
    def minimumLengthEncoding(self, words: List[str]) -> int:
        ...
```

Ghi chú: đề nghe phức tạp ("chuỗi tham chiếu", "chỉ số") nhưng thực chất là: **mỗi từ là hậu tố của một từ khác thì không cần ghi riêng**. Đáp án là tổng `(độ dài + 1)` của những từ **không** là hậu tố của từ nào khác (đã khử trùng). Notebook đi từ kiểm tra từng cặp, tập hợp (xóa hậu tố), Trie đảo ngược, sắp xếp theo chuỗi đảo ngược, tới việc **dựng thật** chuỗi mã hóa để kiểm chứng.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một **mã hóa hợp lệ** gồm chuỗi tham chiếu `s` kết thúc bằng `#` và các chỉ số `indices` sao cho `words[i]` là đoạn của `s` từ `indices[i]` tới ký tự `#` kế tiếp; tìm **độ dài ngắn nhất** của `s`.
2. Nắm các công cụ Python cần thiết: `str.endswith`, cắt chuỗi lấy hậu tố `w[i:]`, `set` và khử trùng, chuỗi đảo ngược `w[::-1]`, `sorted` với khóa tùy ý, và Trie bằng `dict`.
3. Hiểu ý chính: chuỗi tham chiếu ngắn nhất là ghép các từ "tối đại" (không là hậu tố của từ khác) cùng dấu `#`; mỗi từ khác được tìm thấy như một **hậu tố** của một từ tối đại.
4. Đi qua 5 cách: so sánh từng cặp, xóa hậu tố khỏi tập hợp, Trie của các từ đảo ngược, sắp xếp theo chuỗi đảo ngược, và **dựng thật** chuỗi mã hóa (cùng `indices`) để chứng minh tính hợp lệ.
5. Tránh các lỗi kinh điển: không khử trùng các từ giống nhau, nhầm hậu tố với chuỗi con (`in`), quên `+ 1` cho dấu `#`, xóa tiền tố thay vì hậu tố, sắp xếp theo thứ tự thường thay vì theo chuỗi đảo ngược.

## 2. Đề bài & ví dụ

Một **mã hóa hợp lệ** của mảng `words` là một chuỗi tham chiếu `s` và mảng chỉ số `indices` thỏa:

- `words.length == indices.length`.
- Chuỗi tham chiếu `s` kết thúc bằng ký tự `'#'`.
- Với mỗi `indices[i]`, **chuỗi con của `s` bắt đầu tại `indices[i]` và dừng ngay trước ký tự `'#'` kế tiếp** bằng `words[i]`.

Cho mảng `words`, trả về **độ dài của chuỗi tham chiếu ngắn nhất** `s` có thể có của một mã hóa hợp lệ.

**Ví dụ 1**

- Input: `words = ["time", "me", "bell"]`
- Output: `10`
- Giải thích: mã hóa hợp lệ là `s = "time#bell#"` với `indices = [0, 2, 5]`. `words[0] = "time"` bắt đầu tại 0 tới `#`; `words[1] = "me"` bắt đầu tại 2 (`ti|me#`); `words[2] = "bell"` bắt đầu tại 5.

**Ví dụ 2**

- Input: `words = ["t"]`
- Output: `2`
- Giải thích: `s = "t#"`, `indices = [0]`.

**Ràng buộc:**

- `1 <= words.length <= 2000`
- `1 <= words[i].length <= 7`
- `words[i]` chỉ gồm chữ cái thường.

Lưu ý: các từ trong `words` **có thể trùng nhau** (đề không cấm).

In [1]:
from typing import List, Optional, Tuple
from itertools import permutations, combinations
import random, time

examples = [
    (["time", "me", "bell"], 10),
    (["t"], 2),
]
for words, expected in examples:
    print(f"words={words} kỳ vọng={expected}")

words=['time', 'me', 'bell'] kỳ vọng=10
words=['t'] kỳ vọng=2


## 3. Kiến thức nền

### 3.1. Hậu tố (suffix) và `endswith`

**Hậu tố** của chuỗi là phần **cuối** của nó. `"me"` là hậu tố của `"time"` (`"time"` kết thúc bằng `"me"`), nhưng `"im"` **không** phải hậu tố của `"time"` dù là chuỗi con. `w.endswith(x)` kiểm tra `x` có là hậu tố của `w`. Mọi chuỗi đều là hậu tố của chính nó.

In [2]:
w = "time"
for x in ["time", "ime", "me", "e", "im", "ti", ""]:
    print(f"{x!r:7} hậu tố của {w!r}? {w.endswith(x)!s:5} | là chuỗi con? {x in w}")

'time'  hậu tố của 'time'? True  | là chuỗi con? True
'ime'   hậu tố của 'time'? True  | là chuỗi con? True
'me'    hậu tố của 'time'? True  | là chuỗi con? True
'e'     hậu tố của 'time'? True  | là chuỗi con? True
'im'    hậu tố của 'time'? False | là chuỗi con? True
'ti'    hậu tố của 'time'? False | là chuỗi con? True
''      hậu tố của 'time'? True  | là chuỗi con? True


### 3.2. Lấy mọi hậu tố thật sự của một từ

`w[i:]` với `i = 1..len(w)-1` là các hậu tố **thật sự** (khác chính `w`, không rỗng).

In [3]:
w = "time"
print([w[i:] for i in range(1, len(w))])
print("số hậu tố thật sự của từ dài L là L - 1:", len(w) - 1)

['ime', 'me', 'e']
số hậu tố thật sự của từ dài L là L - 1: 3


### 3.3. `set`: khử trùng và xóa phần tử

`set(words)` khử các từ trùng nhau. `s.discard(x)` xóa `x` khỏi `s` nếu có (không lỗi khi không có). Dùng để "xóa các hậu tố của từng từ ra khỏi tập".

In [4]:
words = ["time", "me", "bell", "time"]
s = set(words)
print("sau khi khử trùng:", sorted(s))
s.discard("me")
s.discard("không có")
print(sorted(s))

sau khi khử trùng: ['bell', 'me', 'time']
['bell', 'time']


### 3.4. Chuỗi đảo ngược và sắp xếp theo khóa

`w[::-1]` là chuỗi đảo ngược. **Hậu tố của `w` chính là tiền tố của `w[::-1]`** (đảo ngược). `sorted(xs, key=...)` sắp xếp theo khóa tùy ý; sắp xếp các chuỗi đảo ngược theo thứ tự từ điển làm các từ có chung hậu tố đứng **cạnh nhau**.

In [5]:
words = ["time", "me", "bell", "e", "ell"]
rev = sorted(w[::-1] for w in words)
print(rev)
print("me đảo là", "me"[::-1], "-> là tiền tố của", "time"[::-1], "?", "time"[::-1].startswith("me"[::-1]))

['e', 'em', 'emit', 'lle', 'lleb']
me đảo là em -> là tiền tố của emit ? True


### 3.4b. Trie bằng `dict`

Trie (cây tiền tố) lưu các chuỗi sao cho các chuỗi chung tiền tố dùng chung một đường đi. Ta có thể biểu diễn mỗi nút bằng một `dict` ký tự -> nút con. Nút **không có con** là nút lá.

In [6]:
def trie_insert(root, word):
    node = root
    for ch in word:
        node = node.setdefault(ch, {})
    return node

root = {}
for w in ["emit", "em"]:                       # "time" và "me" đảo ngược
    trie_insert(root, w)
print(root)
def leaves(node, depth=0):
    if not node:
        return [depth]
    return [d for child in node.values() for d in leaves(child, depth + 1)]
print("độ sâu các lá:", leaves(root))

{'e': {'m': {'i': {'t': {}}}}}
độ sâu các lá: [4]


### 3.5. Kiểm tra một mã hóa có hợp lệ không

Cho `s` và `indices`, `words[i]` phải bằng `s[indices[i]:]` cắt tới `#` đầu tiên. Ta cần hàm kiểm tra để **chứng minh** một mã hóa (ở Cách 5) là hợp lệ, và để định nghĩa oracle.

In [7]:
def word_at(s: str, i: int) -> str:
    j = s.index("#", i)
    return s[i:j]

def valid_encoding(words: List[str], s: str, indices: List[int]) -> bool:
    return (s.endswith("#") and len(words) == len(indices)
            and all(0 <= i < len(s) and word_at(s, i) == w for w, i in zip(words, indices)))

print(valid_encoding(["time", "me", "bell"], "time#bell#", [0, 2, 5]))
print(valid_encoding(["time", "me", "bell"], "time#bell#", [0, 3, 5]))     # "e" không phải "me"
print(valid_encoding(["t"], "t#", [0]))

True
False
True


## 4. Ý tưởng nền tảng / trực giác (tính tay)

`words = ["time", "me", "bell"]`:

- `"me"` là hậu tố của `"time"`: nếu đã ghi `"time#"`, từ `"me"` được đọc từ chỉ số 2 (`ti|me#`), **không cần ghi riêng**.
- `"time"` và `"bell"` không là hậu tố của từ nào khác, nên phải ghi riêng: `"time#"` (5 ký tự) và `"bell#"` (5 ký tự).

Độ dài ngắn nhất `= 5 + 5 = 10`.

Quy tắc: **một từ chỉ cần ghi riêng nếu nó không là hậu tố của (từ khác) nào**; mỗi từ phải ghi riêng đóng góp `độ dài + 1` (dấu `#`).

Với từ trùng nhau (`["time", "time"]`): chỉ cần ghi một lần (`"time#"`, `indices = [0, 0]`), nên **phải khử trùng** trước khi xét.

In [8]:
def needs_own(w, others):
    return not any(o != w and o.endswith(w) for o in others)

words = ["time", "me", "bell"]
for w in words:
    print(f"{w!r:7} cần ghi riêng? {needs_own(w, words)}")
print("tổng =", sum(len(w) + 1 for w in words if needs_own(w, words)))

'time'  cần ghi riêng? True
'me'    cần ghi riêng? False
'bell'  cần ghi riêng? True
tổng = 10


## 5. Cách 1 — Brute Force: so sánh từng cặp

**Ý tưởng.** Khử trùng bằng `set`. Với mỗi từ `w`, kiểm tra xem có từ `o` khác (`o != w`) mà `o.endswith(w)` không. Nếu có thì `w` được "phủ" bởi `o`, bỏ qua; nếu không thì cộng `len(w) + 1`.

**Oracle độc lập.** Để chắc đáp án đúng ngay cả khi lập luận hậu tố sai, ta cũng viết oracle **theo đúng định nghĩa**: thử mọi tập con các từ (đã khử trùng) và mọi thứ tự ghép `"w#"`, lấy chuỗi tham chiếu ngắn nhất mà **mã hóa hợp lệ** (mỗi từ tìm được tại một chỉ số). Chỉ dùng cho vài từ.

In [9]:
def encoding_pairs(words: List[str]) -> int:
    uniq = set(words)
    total = 0
    for w in uniq:
        if not any(o != w and o.endswith(w) for o in uniq):
            total += len(w) + 1
    return total

def has_valid_indices(words: List[str], s: str) -> bool:
    """Mỗi từ có đọc được từ một chỉ số nào đó của s (dừng tại '#' kế tiếp) không?"""
    pieces = set()
    for i in range(len(s)):
        j = s.find("#", i)
        if j != -1:
            pieces.add(s[i:j])
    return all(w in pieces for w in words)

def oracle(words: List[str]) -> int:
    uniq = sorted(set(words))
    best = None
    for r in range(1, len(uniq) + 1):
        for subset in combinations(uniq, r):
            for order in permutations(subset):
                s = "".join(w + "#" for w in order)
                if (best is None or len(s) < best) and has_valid_indices(words, s):
                    best = len(s)
    return best

for words, expected in examples:
    assert encoding_pairs(words) == expected == oracle(words)
print("so cặp OK; oracle (định nghĩa) khớp")

so cặp OK; oracle (định nghĩa) khớp


**Độ phức tạp và nút thắt.** Với `n` từ khác nhau, độ dài `L <= 7`: `O(n^2 * L)` (mỗi cặp một lần `endswith`). Với `n = 2000` là `4 * 10^6` lần gọi — chạy được, nhưng ta so sánh **mọi cặp** dù chỉ cần biết "từ này có là hậu tố của từ nào khác". Vì độ dài từ rất ngắn (`<= 7`), có cách khác tốt hơn: **liệt kê hậu tố của từng từ** (tối đa 6) và tra trong tập hợp.

In [10]:
rng = random.Random(1)
for n in (250, 500, 1000, 2000):
    words = ["".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(7)) for _ in range(n)]
    t0 = time.perf_counter()
    encoding_pairs(words)
    print(f"n={n:5}: so từng cặp mất {1000 * (time.perf_counter() - t0):8.1f} ms (số cặp {n * n:,})")

n=  250: so từng cặp mất      6.6 ms (số cặp 62,500)
n=  500: so từng cặp mất     28.4 ms (số cặp 250,000)
n= 1000: so từng cặp mất     99.9 ms (số cặp 1,000,000)


n= 2000: so từng cặp mất    308.3 ms (số cặp 4,000,000)


## 6. Ý tưởng thuật toán / bước đột phá

**Xóa các hậu tố thật sự khỏi tập.** Một từ `w` bị "phủ" khi nó là **hậu tố thật sự của một từ khác `o`**. Đảo ngược cách nhìn: với mỗi từ `o`, mọi hậu tố thật sự `o[i:]` (`i = 1..len(o)-1`) **đều được phủ**, nên xóa chúng khỏi tập. Mỗi từ chỉ có tối đa 6 hậu tố thật sự (`L <= 7`), nên tổng `O(n * L^2)`, độc lập với `n^2`.

Các cách nhìn tương đương:

| Cách | Ý tưởng |
|---|---|
| 2 | Tập hợp: xóa mọi hậu tố thật sự của mỗi từ, cộng `len + 1` các từ còn lại |
| 3 | Trie của các từ **đảo ngược**: mỗi từ tối đại là một lá; cộng `độ sâu + 1` cho từng lá |
| 4 | Sắp xếp các từ đảo ngược theo từ điển; từ `i` bị phủ khi chuỗi đảo ngược của nó là tiền tố của từ kế tiếp |
| 5 | Dựng thật chuỗi mã hóa: duyệt từ dài tới ngắn, chỉ thêm `w#` nếu `w#` chưa là hậu tố của chuỗi đã dựng; trả luôn `indices` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tập hợp và xóa hậu tố

**Giải thích từng đoạn.**

- `good = set(words)` (khử trùng).
- Với mỗi `w` trong `words`: với `k` từ `1` tới `len(w) - 1`, `good.discard(w[k:])`.
- Đáp án: `sum(len(w) + 1 for w in good)`.

Lưu ý: ta xóa theo **mọi** từ gốc (kể cả từ chính nó bị xóa bởi từ khác). Một từ chỉ bị xóa khi nó là hậu tố **thật sự** của từ khác, và `discard` chỉ tác động tới hậu tố khác chính từ.

In [11]:
def encoding_set(words: List[str]) -> int:
    good = set(words)
    for w in words:
        for k in range(1, len(w)):
            good.discard(w[k:])
    return sum(len(w) + 1 for w in good)

for words, expected in examples:
    assert encoding_set(words) == expected
print("tập hợp OK")

tập hợp OK


**Code nộp LeetCode (Cách 2):**

In [12]:
class Solution:
    def minimumLengthEncoding(self, words: List[str]) -> int:
        good = set(words)
        for w in words:
            for k in range(1, len(w)):
                good.discard(w[k:])
        return sum(len(w) + 1 for w in good)

sol_set = Solution().minimumLengthEncoding
print([sol_set(w) for w, _ in examples])

[10, 2]


**Độ phức tạp (Cách 2):** thời gian `O(n * L^2)` (mỗi từ `L - 1` hậu tố, mỗi lần cắt/băm `O(L)`); bộ nhớ `O(n * L)`.

---

### 7.2. Cách 3 — Trie của các từ đảo ngược

**Ý tưởng khác.** Chèn **chuỗi đảo ngược** của mỗi từ vào Trie. Khi đó hậu tố của từ gốc thành **tiền tố** của chuỗi đảo ngược, dùng chung đường đi. Một từ bị phủ khi chuỗi đảo ngược của nó là **tiền tố thật sự** của chuỗi đảo ngược khác, tức nút cuối của nó **không phải lá**. Do đó đáp án = tổng `(độ sâu của lá + 1)` trên các lá.

**Giải thích từng đoạn.**

- `root = {}`; chèn `w[::-1]` bằng `node = node.setdefault(ch, {})`.
- DFS (hoặc đệ quy) trên Trie: tại nút không có con (lá) cộng `depth + 1`.
- Không cần khử trùng: chèn từ trùng không thêm nút.

In [13]:
def encoding_trie(words: List[str]) -> int:
    root = {}
    for w in words:
        node = root
        for ch in reversed(w):
            node = node.setdefault(ch, {})

    def dfs(node, depth):
        if not node:                      # lá: một từ tối đại có độ dài depth
            return depth + 1
        return sum(dfs(child, depth + 1) for child in node.values())

    return dfs(root, 0)

for words, expected in examples:
    assert encoding_trie(words) == expected
print("Trie OK")

Trie OK


**Code nộp LeetCode (Cách 3):** dùng mảng 26 con như lời giải chuẩn.

In [14]:
class Trie:
    def __init__(self) -> None:
        self.children = [None] * 26


class Solution:
    def minimumLengthEncoding(self, words: List[str]) -> int:
        root = Trie()
        for w in words:
            cur = root
            for c in w[::-1]:
                idx = ord(c) - ord("a")
                if cur.children[idx] is None:
                    cur.children[idx] = Trie()
                cur = cur.children[idx]
        return self.dfs(root, 1)

    def dfs(self, cur: Trie, l: int) -> int:
        isLeaf, ans = True, 0
        for i in range(26):
            if cur.children[i] is not None:
                isLeaf = False
                ans += self.dfs(cur.children[i], l + 1)
        if isLeaf:
            ans += l
        return ans

sol_trie = Solution().minimumLengthEncoding
print([sol_trie(w) for w, _ in examples])

[10, 2]


**Độ phức tạp (Cách 3):** thời gian `O(tổng độ dài các từ)` để dựng Trie và `O(số nút * 26)` để duyệt; bộ nhớ `O(số nút * 26)`.

---

### 7.3. Cách 4 — Sắp xếp theo chuỗi đảo ngược

**Ý tưởng khác.** Sắp xếp các **chuỗi đảo ngược** (đã khử trùng) theo thứ tự từ điển. Nếu `a` là tiền tố thật sự của `b` thì `a < b` và **mọi chuỗi nằm giữa `a` và `b` trong thứ tự cũng bắt đầu bằng `a`**; do đó nếu `a` là tiền tố của **một** chuỗi nào thì nó cũng là tiền tố của chuỗi **đứng ngay sau nó**. Vậy chỉ cần so mỗi chuỗi với chuỗi kế tiếp.

**Giải thích từng đoạn.**

- `rev = sorted({w[::-1] for w in words})`.
- Với `i` từ `0` tới `len(rev) - 1`: nếu `i + 1 < len(rev)` và `rev[i + 1].startswith(rev[i])` thì từ `i` bị phủ (bỏ qua); ngược lại cộng `len(rev[i]) + 1`.

In [15]:
def encoding_sort(words: List[str]) -> int:
    rev = sorted({w[::-1] for w in words})
    total = 0
    for i, r in enumerate(rev):
        if i + 1 < len(rev) and rev[i + 1].startswith(r):
            continue
        total += len(r) + 1
    return total

for words, expected in examples:
    assert encoding_sort(words) == expected
print("sắp xếp OK")

sắp xếp OK


**Code nộp LeetCode (Cách 4):**

In [16]:
class Solution:
    def minimumLengthEncoding(self, words: List[str]) -> int:
        rev = sorted({w[::-1] for w in words})
        ans = 0
        for i, r in enumerate(rev):
            if i + 1 < len(rev) and rev[i + 1].startswith(r):
                continue
            ans += len(r) + 1
        return ans

sol_sort = Solution().minimumLengthEncoding
print([sol_sort(w) for w, _ in examples])

[10, 2]


**Độ phức tạp (Cách 4):** thời gian `O(n log n * L)` (sắp xếp chuỗi dài `L`); bộ nhớ `O(n * L)`.

---

### 7.4. Cách 5 — Dựng thật chuỗi mã hóa (kèm `indices`)

**Ý tưởng khác.** Thay vì chỉ đếm, **dựng** mã hóa: duyệt các từ từ **dài tới ngắn** (đã khử trùng). Với từ `w`: nếu `w + "#"` đã là **hậu tố** của chuỗi tham chiếu hiện tại `s` thì `w` đã được phủ, chỉ số là `len(s) - len(w) - 1`; ngược lại thêm `w + "#"` vào cuối `s` (chỉ số là vị trí bắt đầu mới).

Vì sao duyệt dài tới ngắn: từ dài chỉ có thể phủ từ ngắn hơn (không bao giờ ngược lại), và khi ta tới `w`, mọi từ có thể phủ `w` đã được xử lý.

**Giải thích từng đoạn.**

- `s` bắt đầu rỗng; `pos[w]` ghi chỉ số của từng từ.
- Với `w`: `idx = s.find(w + "#")`. Dấu `#` chỉ xuất hiện ở **cuối** một đoạn, nên nếu `w#` xuất hiện trong `s` thì `w` chính là hậu tố của đoạn đó, tức `w` đọc được tại `idx`. Nếu `idx == -1` thì thêm `w + "#"` vào cuối `s` và `idx = len(s)` cũ.
- Trả `len(s)` cùng `indices` theo thứ tự `words` gốc.

In [17]:
def build_encoding(words: List[str]) -> Tuple[int, str, List[int]]:
    s = ""
    pos = {}
    for w in sorted(set(words), key=len, reverse=True):
        idx = s.find(w + "#")
        if idx == -1:
            idx = len(s)
            s += w + "#"
        pos[w] = idx
    return len(s), s, [pos[w] for w in words]

def encoding_build(words: List[str]) -> int:
    return build_encoding(words)[0]

for words, expected in examples:
    n, s, indices = build_encoding(words)
    print(words, "->", n, repr(s), indices, "hợp lệ:", valid_encoding(words, s, indices))
    assert n == expected and valid_encoding(words, s, indices)

['time', 'me', 'bell'] -> 10 'time#bell#' [0, 2, 5] hợp lệ: True
['t'] -> 2 't#' [0] hợp lệ: True


**Code nộp LeetCode (Cách 5):**

In [18]:
class Solution:
    def minimumLengthEncoding(self, words: List[str]) -> int:
        s = ""
        for w in sorted(set(words), key=len, reverse=True):
            if (w + "#") not in s:
                s += w + "#"
        return len(s)

sol_build = Solution().minimumLengthEncoding
print([sol_build(w) for w, _ in examples])

[10, 2]


**Độ phức tạp (Cách 5):** thời gian `O(n * len(s))` cho các phép `in` trên chuỗi dài tới `16000` ký tự (`2000 * 16000 = 3,2 * 10^7` so sánh ký tự trong C, nhanh); bộ nhớ `O(len(s))`. Ưu điểm: cho ra **bằng chứng** (chuỗi và chỉ số).

## 8. TRACE từng bước

### 8.1. Cách 2 trên `["time", "me", "bell"]`

| từ `w` | các hậu tố thật sự `w[k:]` | `good` sau khi xóa |
|---|---|---|
| (khởi đầu) | | `{time, me, bell}` |
| `time` | `ime`, `me`, `e` | `{time, bell}` (xóa `me`) |
| `me` | `e` | `{time, bell}` |
| `bell` | `ell`, `ll`, `l` | `{time, bell}` |

Đáp án `(4 + 1) + (4 + 1) = 10`.

In [19]:
def trace_set(words):
    good = set(words)
    print("khởi đầu:", sorted(good))
    for w in words:
        suffixes = [w[k:] for k in range(1, len(w))]
        removed = [x for x in suffixes if x in good]
        for x in suffixes:
            good.discard(x)
        print(f"  từ {w!r}: hậu tố thật sự {suffixes}, xóa được {removed} -> {sorted(good)}")
    total = sum(len(w) + 1 for w in good)
    print("còn lại", sorted(good), "-> tổng", " + ".join(str(len(w) + 1) for w in sorted(good)), "=", total)
    return total

assert trace_set(["time", "me", "bell"]) == 10
print()
assert trace_set(["time", "time", "me"]) == 5

khởi đầu: ['bell', 'me', 'time']
  từ 'time': hậu tố thật sự ['ime', 'me', 'e'], xóa được ['me'] -> ['bell', 'time']
  từ 'me': hậu tố thật sự ['e'], xóa được [] -> ['bell', 'time']
  từ 'bell': hậu tố thật sự ['ell', 'll', 'l'], xóa được [] -> ['bell', 'time']
còn lại ['bell', 'time'] -> tổng 5 + 5 = 10

khởi đầu: ['me', 'time']
  từ 'time': hậu tố thật sự ['ime', 'me', 'e'], xóa được ['me'] -> ['time']
  từ 'time': hậu tố thật sự ['ime', 'me', 'e'], xóa được [] -> ['time']
  từ 'me': hậu tố thật sự ['e'], xóa được [] -> ['time']
còn lại ['time'] -> tổng 5 = 5


### 8.2. Cách 3: Trie của các từ đảo ngược

Chuỗi đảo ngược: `emit`, `em`, `lleb`. Trie: đường `e -> m -> i -> t` (lá, độ sâu 4) chứa luôn `e -> m` (không phải lá vì có con `i`); đường `l -> l -> e -> b` (lá, độ sâu 4). Hai lá: `(4 + 1) + (4 + 1) = 10`.

In [20]:
def trace_trie(words):
    root = {}
    for w in words:
        node = root
        for ch in reversed(w):
            node = node.setdefault(ch, {})
    def show(node, depth=0):
        for ch, child in sorted(node.items()):
            tag = "  <- LÁ (từ tối đại, độ dài %d, cộng %d)" % (depth + 1, depth + 2) if not child else ""
            print("  " * depth + ch + tag)
            show(child, depth + 1)
    show(root)
    return encoding_trie(words)

assert trace_trie(["time", "me", "bell"]) == 10

e
  m
    i
      t  <- LÁ (từ tối đại, độ dài 4, cộng 5)
l
  l
    e
      b  <- LÁ (từ tối đại, độ dài 4, cộng 5)


### 8.3. Cách 4: sắp xếp theo chuỗi đảo ngược

`rev = sorted(["emit", "em", "lleb"]) = ["em", "emit", "lleb"]`. So từng chuỗi với chuỗi kế tiếp: `"emit".startswith("em")` -> bỏ `em`; `"lleb".startswith("emit")`? không -> cộng `5`; `lleb` là cuối -> cộng `5`. Tổng 10.

In [21]:
def trace_sort(words):
    rev = sorted({w[::-1] for w in words})
    print("rev đã sắp xếp:", rev)
    total = 0
    for i, r in enumerate(rev):
        nxt = rev[i + 1] if i + 1 < len(rev) else None
        covered = nxt is not None and nxt.startswith(r)
        if covered:
            print(f"  {r!r}: chuỗi kế tiếp {nxt!r} bắt đầu bằng {r!r} -> bị phủ")
        else:
            total += len(r) + 1
            print(f"  {r!r}: chuỗi kế tiếp {nxt!r} không bắt đầu bằng {r!r} -> cộng {len(r) + 1}")
    print("tổng =", total)
    return total

assert trace_sort(["time", "me", "bell"]) == 10
print()
assert trace_sort(["a", "ba", "cba", "x"]) == 6

rev đã sắp xếp: ['em', 'emit', 'lleb']
  'em': chuỗi kế tiếp 'emit' bắt đầu bằng 'em' -> bị phủ
  'emit': chuỗi kế tiếp 'lleb' không bắt đầu bằng 'emit' -> cộng 5
  'lleb': chuỗi kế tiếp None không bắt đầu bằng 'lleb' -> cộng 5
tổng = 10

rev đã sắp xếp: ['a', 'ab', 'abc', 'x']
  'a': chuỗi kế tiếp 'ab' bắt đầu bằng 'a' -> bị phủ
  'ab': chuỗi kế tiếp 'abc' bắt đầu bằng 'ab' -> bị phủ
  'abc': chuỗi kế tiếp 'x' không bắt đầu bằng 'abc' -> cộng 4
  'x': chuỗi kế tiếp None không bắt đầu bằng 'x' -> cộng 2
tổng = 6


### 8.4. Cách 5: dựng chuỗi và `indices`

Duyệt từ dài tới ngắn: `time` (4), `bell` (4), `me` (2). `s = ""`: `"time#"` chưa có -> thêm, chỉ số 0; `"bell#"` chưa có -> thêm, chỉ số 5; `"me#"` đã nằm trong `"time#bell#"` (ở vị trí 2) -> chỉ số 2. Kết quả `s = "time#bell#"`, `indices = [0, 2, 5]`, đúng như đề.

In [22]:
def trace_build(words):
    s = ""
    pos = {}
    for w in sorted(set(words), key=len, reverse=True):
        idx = s.find(w + "#")
        if idx == -1:
            idx = len(s)
            s += w + "#"
            print(f"  {w!r}: chưa có {w + '#'!r} trong s -> thêm, chỉ số {idx}; s = {s!r}")
        else:
            print(f"  {w!r}: {w + '#'!r} đã nằm trong s tại chỉ số {idx} -> dùng lại")
        pos[w] = idx
    indices = [pos[w] for w in words]
    print("kết quả: s =", repr(s), "indices =", indices, "độ dài", len(s))
    assert valid_encoding(words, s, indices)
    return len(s)

assert trace_build(["time", "me", "bell"]) == 10

  'time': chưa có 'time#' trong s -> thêm, chỉ số 0; s = 'time#'
  'bell': chưa có 'bell#' trong s -> thêm, chỉ số 5; s = 'time#bell#'
  'me': 'me#' đã nằm trong s tại chỉ số 2 -> dùng lại
kết quả: s = 'time#bell#' indices = [0, 2, 5] độ dài 10


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (tập hợp + xóa hậu tố): ngắn, `O(n L^2)` với `L <= 7`, ít chỗ sai.

```python
class Solution:
    def minimumLengthEncoding(self, words: List[str]) -> int:
        good = set(words)
        for w in words:
            for k in range(1, len(w)):
                good.discard(w[k:])
        return sum(len(w) + 1 for w in good)
```

In [23]:
class Solution:
    def minimumLengthEncoding(self, words: List[str]) -> int:
        good = set(words)
        for w in words:
            for k in range(1, len(w)):
                good.discard(w[k:])
        return sum(len(w) + 1 for w in good)

sol = Solution()
for words, expected in examples:
    got = sol.minimumLengthEncoding(words)
    print(words, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

['time', 'me', 'bell'] -> 10 OK
['t'] -> 2 OK


## 10. Vì sao đúng

**(a) Cận dưới.** Gọi từ `w` (trong tập đã khử trùng) là **tối đại** nếu **không** có từ khác nào nhận `w` làm hậu tố. Trong một mã hóa hợp lệ, mỗi từ `w` được đọc tại một chỉ số `i`: `s[i:j] = w` với `s[j] = '#'`. Gọi **đoạn** là phần của `s` giữa hai dấu `#` liên tiếp (hoặc từ đầu tới `#` đầu tiên); `w` là hậu tố của đoạn chứa `j`. Hai từ tối đại khác nhau `m1`, `m2` **không thể cùng là hậu tố của một đoạn** (nếu cùng là hậu tố của một đoạn thì hậu tố ngắn hơn là hậu tố của hậu tố dài hơn, tức một trong hai không tối đại). Vậy các từ tối đại cần các đoạn **khác nhau**; đoạn chứa `m` có độ dài `>= len(m)` cộng một dấu `#`. Do đó `|s| >= sum over từ tối đại (len + 1)`.

**(b) Cận trên.** Chuỗi `s = "m1#m2#...#"` (ghép mọi từ tối đại) là mã hóa hợp lệ: mỗi từ `w` không tối đại là hậu tố thật sự của một từ `w1` trong `words`; nếu `w1` còn không tối đại thì lặp lại với `w1` (độ dài tăng ngặt, hậu tố của hậu tố là hậu tố) cho tới một từ tối đại `m` mà `w` là hậu tố; vậy `w` đọc được tại chỉ số `len(m) - len(w)` bên trong `m#`. Độ dài `s = sum(len(m) + 1)`.

Hai chiều cho `min |s| = sum over từ tối đại (len + 1)`.

**(c) Các cách đo "tối đại".** Cách 1: `không tồn tại o != w với o.endswith(w)` là định nghĩa. Cách 2: xóa mọi hậu tố thật sự của mọi từ ⇒ phần còn lại chính xác là các từ không phải hậu tố thật sự của từ nào. Cách 3: lá của Trie ứng với chuỗi đảo ngược không là tiền tố thật sự của chuỗi khác. Cách 4: nếu `a` là tiền tố thật sự của một chuỗi `b` trong danh sách đã sắp xếp thì `a <= c <= b` kéo theo `c` bắt đầu bằng `a` với mọi `c` ở giữa; đặc biệt phần tử đứng ngay sau `a` bắt đầu bằng `a`. Cách 5: duyệt từ dài tới ngắn nên mọi từ có thể phủ `w` đã có mặt trong `s` khi tới `w`; `(w + "#") in s` đúng khi `w` là hậu tố của một đoạn đã ghi (vì `w#` kết thúc ở `#` nghĩa là `w` nằm cuối một đoạn).

**Kiểm chứng bằng máy:** so mọi cách với oracle theo định nghĩa (thử mọi tập con và thứ tự ghép) trên **mọi** danh sách từ nhỏ (bảng chữ cái `{a, b}`, độ dài từ tới 3, tới 4 từ), và kiểm tra mọi `(s, indices)` do Cách 5 dựng là mã hóa hợp lệ trên mọi input ngẫu nhiên.

In [24]:
solutions = {
    "pairs": encoding_pairs,
    "set": encoding_set,
    "trie": encoding_trie,
    "sort": encoding_sort,
    "build": encoding_build,
    "sol_set": sol_set,
    "sol_trie": sol_trie,
    "sol_sort": sol_sort,
    "sol_build": sol_build,
    "final": Solution().minimumLengthEncoding,
}

def all_strings(alphabet: str, max_len: int):
    out, frontier = [], [""]
    for _ in range(max_len):
        frontier = [x + c for x in frontier for c in alphabet]
        out += frontier
    return [x for x in out if x]

pool = all_strings("ab", 3)                         # 14 từ
checked = 0
for r in range(1, 5):
    for words in combinations(pool, r):
        words = list(words)
        ref = oracle(words)
        for name, fn in solutions.items():
            assert fn(words) == ref, (name, words, fn(words), ref)
        checked += 1
print("mọi tập 1..4 từ khác nhau từ 14 từ {a,b} độ dài <= 3:", checked, "tập x", len(solutions), "cài đặt khớp oracle theo định nghĩa")

mọi tập 1..4 từ khác nhau từ 14 từ {a,b} độ dài <= 3: 1470 tập x 10 cài đặt khớp oracle theo định nghĩa


## 11. Time / Space Complexity

Gọi `n = len(words)` (`<= 2000`), `L = 7` là độ dài từ tối đa.

| Cách | Time | Space |
|---|---|---|
| 1. So từng cặp | `O(n^2 L)` | `O(n L)` |
| 2. Tập hợp, xóa hậu tố | `O(n L^2)` | `O(n L)` |
| 3. Trie đảo ngược | `O(n L)` + duyệt Trie | `O(n L)` (mảng 26: `O(26 n L)`) |
| 4. Sắp xếp theo chuỗi đảo ngược | `O(n L log n)` | `O(n L)` |
| 5. Dựng chuỗi mã hóa | `O(n * |s|)` (so khớp trong C) | `O(|s|)`, `|s| <= 8n` |

Đo thử với 2000 từ dài 7:

In [25]:
rng = random.Random(820)
alphabet = "abcdefghijklmnopqrstuvwxyz"
tests = []
tests.append(["".join(rng.choice(alphabet) for _ in range(7)) for _ in range(2000)])          # gần như không có hậu tố chung
tests.append(["".join(rng.choice("ab") for _ in range(rng.randint(1, 7))) for _ in range(2000)])  # nhiều quan hệ hậu tố
base = ["".join(rng.choice(alphabet) for _ in range(7)) for _ in range(300)]
tests.append([w[k:] for w in base for k in range(7)][:2000])                                    # mọi hậu tố của 300 từ
tests.append(["a" * k for k in range(1, 8)] * 285)                                              # trùng lặp nhiều

ref = None
for name in ["pairs", "set", "trie", "sort", "build"]:
    t0 = time.perf_counter()
    res = [solutions[name](w) for w in tests]
    dt = (time.perf_counter() - t0) * 1000
    if ref is None:
        ref = res
    assert res == ref, name
    print(f"{name:7} {len(tests)} test x 2000 từ: {dt:9.1f} ms")
print("kết quả:", ref, "(tối đa 2000 * 8 =", 2000 * 8, ")")

pairs   4 test x 2000 từ:     455.8 ms
set     4 test x 2000 từ:       3.7 ms
trie    4 test x 2000 từ:       8.6 ms
sort    4 test x 2000 từ:       2.4 ms
build   4 test x 2000 từ:      12.6 ms
kết quả: [16000, 927, 2288, 8] (tối đa 2000 * 8 = 16000 )


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai, đối chiếu mọi cách với oracle theo định nghĩa, kiểm tra **mọi `(s, indices)` dựng ra đều là mã hóa hợp lệ**, và test ngẫu nhiên có seed (đúng ràng buộc).

In [26]:
# 12.1 ví dụ đề
for words, expected in examples:
    for name, fn in solutions.items():
        assert fn(words) == expected, (name, words)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    (["t"], 2),                                          # một từ một chữ
    (["time", "me", "bell"], 10),                        # ví dụ của đề
    (["time", "time"], 5),                               # từ trùng: ghi một lần
    (["me", "time"], 5),                                 # thứ tự không quan trọng
    (["a", "b"], 4),                                     # hai từ rời nhau
    (["a", "ba", "cba"], 4),                             # chuỗi hậu tố lồng nhau: chỉ ghi "cba#"
    (["time", "im"], 8),                                 # "im" là chuỗi con nhưng KHÔNG là hậu tố -> ghi riêng
    (["ab", "b", "ab"], 3),                              # "b" là hậu tố của "ab"; "ab" trùng
    (["abc", "bc", "c", "ab"], 7),                       # "abc#" (4) + "ab#" (3); "ab" là tiền tố chứ không phải hậu tố
    (["aaaaaaa", "aaaaaa", "aaaaa"], 8),                 # tất cả là hậu tố của từ dài nhất
    (["xyz", "yz", "z", "wxyz", "abc"], 9),              # "wxyz#" (5) + "abc#" (4)
    (["abcdefg"] * 2000, 8),                             # 2000 từ giống nhau
]
for words, expected in edge:
    if len(set(words)) <= 5:
        assert oracle(words) == expected, ("oracle khác kỳ vọng", words, oracle(words))
    for name, fn in solutions.items():
        assert fn(words) == expected, (name, words, fn(words), expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 12 ca


In [27]:
# 12.3 phản ví dụ: các cách sai
def wrong_no_dedupe(words):             # không khử trùng: cộng cho mọi từ không bị phủ, kể cả bản sao
    return sum(len(w) + 1 for w in words if not any(o != w and o.endswith(w) for o in words))

def wrong_substring(words):             # nhầm chuỗi con với hậu tố
    uniq = set(words)
    return sum(len(w) + 1 for w in uniq if not any(o != w and w in o for o in uniq))

def wrong_no_hash(words):               # quên + 1 cho dấu '#'
    uniq = set(words)
    return sum(len(w) for w in uniq if not any(o != w and o.endswith(w) for o in uniq))

def wrong_prefix(words):                # xóa tiền tố thay vì hậu tố
    good = set(words)
    for w in words:
        for k in range(1, len(w)):
            good.discard(w[:k])
    return sum(len(w) + 1 for w in good)

def wrong_plain_sort(words):            # sắp xếp theo thứ tự thường thay vì chuỗi đảo ngược
    uniq = sorted(set(words))
    total = 0
    for i, w in enumerate(uniq):
        if i + 1 < len(uniq) and uniq[i + 1].endswith(w):
            continue
        total += len(w) + 1
    return total

wrongs = [
    ("không khử trùng", wrong_no_dedupe, ["time", "time"]),
    ("nhầm chuỗi con với hậu tố", wrong_substring, ["time", "im"]),
    ("quên + 1 cho dấu #", wrong_no_hash, ["t"]),
    ("xóa tiền tố thay vì hậu tố", wrong_prefix, ["time", "me"]),
    ("sắp xếp thứ tự thường", wrong_plain_sort, ["a", "ab", "ba"]),
]
for name, fn, words in wrongs:
    truth = oracle(words)
    print(f"{name:30} words={words}: sai = {fn(words)}, đúng = {truth}")
    assert fn(words) != truth, name
print("12.3 phản ví dụ: OK (các cách sai đều bị lộ)")

không khử trùng                words=['time', 'time']: sai = 10, đúng = 5
nhầm chuỗi con với hậu tố      words=['time', 'im']: sai = 5, đúng = 8
quên + 1 cho dấu #             words=['t']: sai = 1, đúng = 2
xóa tiền tố thay vì hậu tố     words=['time', 'me']: sai = 8, đúng = 5
sắp xếp thứ tự thường          words=['a', 'ab', 'ba']: sai = 8, đúng = 6
12.3 phản ví dụ: OK (các cách sai đều bị lộ)


In [28]:
# 12.4 mọi (s, indices) do Cách 5 dựng ra là mã hóa hợp lệ, và độ dài khớp đáp án
rng = random.Random(2820)
for _ in range(500):
    n = rng.randint(1, 30)
    alpha = rng.choice(["ab", "abc", "abcdefghijklmnopqrstuvwxyz"])
    words = ["".join(rng.choice(alpha) for _ in range(rng.randint(1, 7))) for _ in range(n)]
    length, s, indices = build_encoding(words)
    assert valid_encoding(words, s, indices), (words, s, indices)
    assert length == len(s) == Solution().minimumLengthEncoding(words)
print("12.4 mọi mã hóa dựng được đều hợp lệ: OK")

# 12.5 ngẫu nhiên nhỏ so với oracle theo định nghĩa (<= 5 từ khác nhau)
rng = random.Random(7)
for _ in range(400):
    k = rng.randint(1, 5)
    alpha = rng.choice(["ab", "abc"])
    words = ["".join(rng.choice(alpha) for _ in range(rng.randint(1, 4))) for _ in range(k)]
    if len(set(words)) > 5:
        continue
    ref = oracle(words)
    for name, fn in solutions.items():
        assert fn(words) == ref, (name, words)
print("12.5 ngẫu nhiên nhỏ so với oracle: OK")

# 12.6 ngẫu nhiên lớn (đúng ràng buộc: tới 2000 từ, mỗi từ 1..7 chữ cái thường): các cách khớp nhau
rng = random.Random(77)
for _ in range(60):
    n = rng.randint(1, 2000)
    alpha = rng.choice(["ab", "abc", "abcdefghij", "abcdefghijklmnopqrstuvwxyz"])
    words = ["".join(rng.choice(alpha) for _ in range(rng.randint(1, 7))) for _ in range(n)]
    res = {name: fn(words) for name, fn in solutions.items() if name != "pairs" or n <= 600}
    assert len(set(res.values())) == 1, (n, res)
    assert 2 <= res["final"] <= 8 * n
print("12.6 ngẫu nhiên lớn: OK")

# 12.7 tính chất: thêm một hậu tố của từ đã có không làm đổi đáp án; thêm từ mới (không là hậu tố) tăng đúng len+1
rng = random.Random(9)
for _ in range(300):
    words = ["".join(rng.choice("abc") for _ in range(rng.randint(2, 7))) for _ in range(rng.randint(1, 8))]
    base = Solution().minimumLengthEncoding(words)
    w = rng.choice(words)
    k = rng.randint(1, len(w) - 1)
    assert Solution().minimumLengthEncoding(words + [w[k:]]) == base
print("12.7 thêm hậu tố của một từ đã có không đổi đáp án: OK")

12.4 mọi mã hóa dựng được đều hợp lệ: OK
12.5 ngẫu nhiên nhỏ so với oracle: OK


12.6 ngẫu nhiên lớn: OK
12.7 thêm hậu tố của một từ đã có không đổi đáp án: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không khử trùng các từ giống nhau

Các từ trùng nhau chỉ cần ghi một lần.

In [29]:
words = ["time", "time"]
print("sai :", wrong_no_dedupe(words), "| đúng:", Solution().minimumLengthEncoding(words))
assert wrong_no_dedupe(words) == 10 and Solution().minimumLengthEncoding(words) == 5

sai : 10 | đúng: 5


Vì sao sai: nếu so từng cặp `o != w` thì hai bản sao của `"time"` đều coi nhau là không phải "từ khác" theo giá trị nên đều được cộng. **Đúng:** `set(words)` trước.

### Lỗi 2 — Nhầm chuỗi con với hậu tố

`"im"` nằm trong `"time"` nhưng `"time#"` không cho đọc `"im"` (sau `im` còn `e`, chưa tới `#`).

In [30]:
words = ["time", "im"]
print("sai (dùng in):", wrong_substring(words), "| đúng:", Solution().minimumLengthEncoding(words))
assert wrong_substring(words) == 5 and Solution().minimumLengthEncoding(words) == 8

sai (dùng in): 5 | đúng: 8


Vì sao sai: từ phải kết thúc ngay tại `#` nên chỉ **hậu tố** mới đọc được. **Đúng:** `endswith`.

### Lỗi 3 — Quên `+ 1` cho dấu `#`

Mỗi từ ghi riêng chiếm `len(w) + 1` ký tự.

In [31]:
words = ["t"]
print("sai :", wrong_no_hash(words), "| đúng:", Solution().minimumLengthEncoding(words))
assert wrong_no_hash(words) == 1 and Solution().minimumLengthEncoding(words) == 2

sai : 1 | đúng: 2


Vì sao sai: chuỗi tham chiếu phải **kết thúc bằng `#`** và mỗi từ cần `#` đánh dấu kết thúc. **Đúng:** `len(w) + 1`.

### Lỗi 4 — Xóa tiền tố thay vì hậu tố

`"ti"` là tiền tố của `"time"` nhưng `"ti#"` không đọc được trong `"time#"`.

In [32]:
words = ["time", "me"]
print("sai (xóa tiền tố):", wrong_prefix(words), "| đúng:", Solution().minimumLengthEncoding(words))
assert wrong_prefix(words) == 8 and Solution().minimumLengthEncoding(words) == 5

sai (xóa tiền tố): 8 | đúng: 5


Vì sao sai: ta cần xóa những từ là **hậu tố** (`w[k:]`), không phải tiền tố (`w[:k]`). **Đúng:** `good.discard(w[k:])`.

### Lỗi 5 — Sắp xếp thứ tự thường thay vì chuỗi đảo ngược

Hai từ có chung hậu tố không đứng cạnh nhau khi sắp xếp theo thứ tự thường.

In [33]:
words = ["a", "ab", "ba"]
print("sai (sắp xếp thường):", wrong_plain_sort(words), "| đúng:", Solution().minimumLengthEncoding(words))
assert wrong_plain_sort(words) == 8 and Solution().minimumLengthEncoding(words) == 6

sai (sắp xếp thường): 8 | đúng: 6


Vì sao sai: theo thứ tự thường `"a" < "ab" < "ba"`: `"ab"` chen giữa `"a"` và `"ba"` nên so hàng xóm bỏ sót việc `"a"` là hậu tố của `"ba"`. Tính chất "hậu tố đứng sát" chỉ đúng với **chuỗi đảo ngược**. **Đúng:** `sorted(w[::-1] for w in words)` và so tiền tố.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "hậu tố chung", "chuỗi ghép ngắn nhất chứa các từ như một phần" -> đảo ngược chuỗi để biến hậu tố thành tiền tố rồi dùng **Trie** hoặc **sắp xếp**.
- **Khử trùng trước** (`set`) khi các phần tử có thể trùng.
- Từ ngắn (`L <= 7`) nên liệt kê hậu tố `O(L)` mỗi từ là rẻ.
- Trie: bài này là "đếm lá" (hoặc tổng độ sâu các lá + 1).
- Sắp xếp: tiền tố của một chuỗi luôn đứng **ngay trước** một chuỗi bắt đầu bằng nó (sau khi sắp xếp), nên chỉ so hàng xóm.
- **Dựng bằng chứng** (chuỗi + chỉ số) giúp chắc chắn thuật toán đúng.
- Cùng họ: Longest Common Prefix (14), Implement Trie (208), Stream of Characters (1032 — Trie đảo ngược), Word Break.
- Test nên có: từ trùng, hậu tố lồng nhau, chuỗi con không phải hậu tố, một từ một chữ.

## 15. Biến thể

1. **Phiên bản tiền tố** (mã hóa theo "bắt đầu bằng"): đáp án là tổng độ dài các từ không là tiền tố của từ khác, `+ 1` mỗi từ nếu vẫn dùng `#`.
2. **Trả về mã hóa** (chuỗi và chỉ số) — đã làm (Cách 5, hàm `build_encoding`).
3. **Số từ phải ghi riêng** (số từ tối đại): `len(good)` ở Cách 2.
4. Bài liên quan: Stream of Characters (1032), Prefix and Suffix Search (745), Longest Word in Dictionary (720).

Dưới đây là code cho biến thể 1 và 3, kiểm bằng oracle.

In [34]:
def minimal_roots(words: List[str]) -> List[str]:
    good = set(words)
    for w in words:
        for k in range(1, len(w)):
            good.discard(w[k:])
    return sorted(good)

def encoding_prefix_version(words: List[str]) -> int:
    good = set(words)
    for w in words:
        for k in range(1, len(w)):
            good.discard(w[:k])
    return sum(len(w) + 1 for w in good)

def oracle_prefix(words):
    uniq = set(words)
    return sum(len(w) + 1 for w in uniq if not any(o != w and o.startswith(w) for o in uniq))

assert minimal_roots(["time", "me", "bell"]) == ["bell", "time"]
rng = random.Random(15)
for _ in range(300):
    words = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(1, 8))]
    assert encoding_prefix_version(words) == oracle_prefix(words), words
    assert len(minimal_roots(words)) == sum(1 for w in set(words) if not any(o != w and o.endswith(w) for o in set(words)))
print("số từ phải ghi riêng của ví dụ 1:", len(minimal_roots(["time", "me", "bell"])))
print("Biến thể: OK")

số từ phải ghi riêng của ví dụ 1: 2
Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. So từng cặp | `endswith` mọi cặp | `O(n^2 L)` | `O(nL)` | Hiểu đề; `n` nhỏ |
| 2. Tập hợp, xóa hậu tố | Xóa `w[k:]` | `O(n L^2)` | `O(nL)` | **Cách nộp chính** khi `L` nhỏ |
| 3. Trie đảo ngược | Đếm lá | `O(nL)` | `O(26nL)` | Khi `L` lớn hoặc cần mở rộng (prefix search) |
| 4. Sắp xếp đảo ngược | So hàng xóm | `O(nL log n)` | `O(nL)` | Không cần cấu trúc dữ liệu đặc biệt |
| 5. Dựng chuỗi mã hóa | Dài -> ngắn, `w#` đã có chưa | `O(n * |s|)` | `O(|s|)` | Cần chứng minh bằng `(s, indices)` |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print([ "abc"[k:] for k in range(1, 3) ])
print("cba".endswith("ba"), "cba".startswith("ba"))
print(Solution().minimumLengthEncoding(["a", "ba", "cba"]))
```

In [35]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `words = ["time", "me", "bell", "ell", "l"]`: liệt kê tập các từ **không** là hậu tố của từ khác (các từ phải ghi riêng) và độ dài ngắn nhất.

In [36]:
# Bài 2: điền kết quả tính tay
hand_roots = None        # list đã sắp xếp
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cách "tập hợp và xóa hậu tố".

In [37]:
def exercise_3_set(words: List[str]) -> int:
    # TODO 1: good = tập các từ (khử trùng)
    # TODO 2: với mỗi từ w, xóa khỏi good mọi hậu tố thật sự w[k:] (k từ 1 tới len(w) - 1)
    # TODO 3: trả về tổng (len(w) + 1) của các từ còn lại trong good
    return None


# assert exercise_3_set(["time", "me", "bell"]) == 10
# assert exercise_3_set(["t"]) == 2
# assert exercise_3_set(["time", "time"]) == 5

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_sort(words)`: sắp xếp các chuỗi đảo ngược đã khử trùng và so mỗi chuỗi với chuỗi kế tiếp.

In [38]:
def exercise_4_sort(words: List[str]) -> int:
    # TODO
    pass


# assert exercise_4_sort(["time", "me", "bell"]) == 10
# assert exercise_4_sort(["t"]) == 2
# assert exercise_4_sort(["a", "ba", "cba"]) == 4

### Bài 5 — Mini challenge: dựng mã hóa ngắn nhất (khó)

Viết `exercise_5_encode(words)` trả về `(s, indices)` là **một** mã hóa hợp lệ có độ dài ngắn nhất (có thể có nhiều đáp án, ví dụ thứ tự các từ trong `s` tùy ý). Kiểm tra bằng `valid_encoding` và độ dài bằng đáp án tối ưu.

```python
exercise_5_encode(["time", "me", "bell"])  -> ("time#bell#", [0, 2, 5])   # hoặc "bell#time#" với indices khác
```
Gợi ý: duyệt từ dài tới ngắn; `s.find(w + "#")`.

In [39]:
def exercise_5_encode(words: List[str]) -> Tuple[str, List[int]]:
    # TODO
    pass


# for words in (["time", "me", "bell"], ["t"], ["a", "ba", "cba", "x"], ["time", "im"]):
#     s, indices = exercise_5_encode(words)
#     assert valid_encoding(words, s, indices)
#     assert len(s) == Solution().minimumLengthEncoding(words)

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `"abc"[1:]` và `"abc"[2:]`.
- Mức 2: `"cba"` kết thúc bằng `"ba"` nhưng không bắt đầu bằng `"ba"`; `"a"`, `"ba"` là hậu tố của `"cba"`.
- Mức 3: `['bc', 'c']`, `True False`, `4`.

**Bài 2**
- Mức 1: `"me"` là hậu tố của `"time"`; `"ell"` là hậu tố của `"bell"`; `"l"` là hậu tố của `"ell"` và `"bell"`.
- Mức 2: còn lại `"time"` và `"bell"`.
- Mức 3: `["bell", "time"]`, độ dài `5 + 5 = 10`.

**Bài 3**
- Mức 1: `good = set(words)`.
- Mức 2: `for w in words: for k in range(1, len(w)): good.discard(w[k:])`.
- Mức 3: `return sum(len(w) + 1 for w in good)`.

**Bài 4**
- Mức 1: `rev = sorted({w[::-1] for w in words})`.
- Mức 2: từ `i` bị phủ nếu `rev[i + 1].startswith(rev[i])`.
- Mức 3: cộng `len(rev[i]) + 1` cho các từ không bị phủ.

**Bài 5**
- Mức 1: `for w in sorted(set(words), key=len, reverse=True)`.
- Mức 2: `idx = s.find(w + "#")`; nếu `-1` thì `idx = len(s)` và `s += w + "#"`.
- Mức 3: lưu `pos[w] = idx` và trả `(s, [pos[w] for w in words])`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [40]:
print(["abc"[k:] for k in range(1, 3)])
print("cba".endswith("ba"), "cba".startswith("ba"))
print(Solution().minimumLengthEncoding(["a", "ba", "cba"]))

assert ["abc"[k:] for k in range(1, 3)] == ["bc", "c"]
assert "cba".endswith("ba") and not "cba".startswith("ba")
assert Solution().minimumLengthEncoding(["a", "ba", "cba"]) == 4

['bc', 'c']
True False
4


### Đáp án Bài 2

Các từ phải ghi riêng: `["bell", "time"]`; độ dài ngắn nhất `5 + 5 = 10` (`"me"`, `"ell"`, `"l"` đều là hậu tố của từ khác).

In [41]:
hand_roots_solution = ["bell", "time"]
hand_answer_solution = 10
words = ["time", "me", "bell", "ell", "l"]
assert minimal_roots(words) == hand_roots_solution
assert Solution().minimumLengthEncoding(words) == hand_answer_solution == oracle(words)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [42]:
def exercise_3_set_solution(words: List[str]) -> int:
    good = set(words)
    for w in words:
        for k in range(1, len(w)):
            good.discard(w[k:])
    return sum(len(w) + 1 for w in good)

assert exercise_3_set_solution(["time", "me", "bell"]) == 10
assert exercise_3_set_solution(["t"]) == 2
assert exercise_3_set_solution(["time", "time"]) == 5
rng = random.Random(3)
for _ in range(200):
    words = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 5))]
    if len(set(words)) <= 5:
        assert exercise_3_set_solution(words) == oracle(words), words
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [43]:
def exercise_4_sort_solution(words: List[str]) -> int:
    rev = sorted({w[::-1] for w in words})
    total = 0
    for i, r in enumerate(rev):
        if i + 1 < len(rev) and rev[i + 1].startswith(r):
            continue
        total += len(r) + 1
    return total

assert exercise_4_sort_solution(["time", "me", "bell"]) == 10
assert exercise_4_sort_solution(["t"]) == 2
assert exercise_4_sort_solution(["a", "ba", "cba"]) == 4
rng = random.Random(4)
for _ in range(200):
    words = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 6))) for _ in range(rng.randint(1, 30))]
    assert exercise_4_sort_solution(words) == encoding_pairs(words), words
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [44]:
def exercise_5_encode_solution(words: List[str]) -> Tuple[str, List[int]]:
    s = ""
    pos = {}
    for w in sorted(set(words), key=len, reverse=True):
        idx = s.find(w + "#")
        if idx == -1:
            idx = len(s)
            s += w + "#"
        pos[w] = idx
    return s, [pos[w] for w in words]

for words in (["time", "me", "bell"], ["t"], ["a", "ba", "cba", "x"], ["time", "im"]):
    s, indices = exercise_5_encode_solution(words)
    assert valid_encoding(words, s, indices)
    assert len(s) == Solution().minimumLengthEncoding(words)
rng = random.Random(5)
for _ in range(300):
    words = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 7))) for _ in range(rng.randint(1, 40))]
    s, indices = exercise_5_encode_solution(words)
    assert valid_encoding(words, s, indices) and len(s) == encoding_pairs(words), words
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (tập hợp, xóa hậu tố thật sự)

```python
class Solution:
    def minimumLengthEncoding(self, words: List[str]) -> int:
        good = set(words)                          # khử trùng
        for w in words:
            for k in range(1, len(w)):             # mọi hậu tố THẬT SỰ w[k:]
                good.discard(w[k:])
        return sum(len(w) + 1 for w in good)       # mỗi từ còn lại: độ dài + '#'
```

Công thức nhớ nhanh: đáp án `= sum(len(w) + 1)` trên các từ (khử trùng) **không là hậu tố của từ khác**.

### Checklist trước khi submit

- [ ] Đã khử trùng (`set`) các từ giống nhau?
- [ ] Dùng **hậu tố** (`w[k:]`, `endswith`), không dùng tiền tố hoặc chuỗi con (`in`)?
- [ ] Hậu tố **thật sự**: `k` từ `1` (không xóa chính từ đó)?
- [ ] Mỗi từ ghi riêng cộng `len(w) + 1` (có dấu `#`)?
- [ ] Nếu dùng sắp xếp: sắp xếp **chuỗi đảo ngược** và so tiền tố với chuỗi kế tiếp?
- [ ] Đã thử: một từ, từ trùng, hậu tố lồng nhau (`a`, `ba`, `cba`), chuỗi con không phải hậu tố (`time`, `im`)?
- [ ] Tên method đúng `minimumLengthEncoding(self, words)` và trả `int`?